In [ ]:
!pip install datasets
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("PortPy-Project/PortPy_Dataset")

In [ ]:
df.head(5)

,patient_id,ct_data,ct_metadata,structureset_data,structureset_metadata,beam_data_paths,beam_metadata_paths,optimization_voxels_data,optimization_voxels_metadata,planner_beams,rt_dose_echo_imrt,rt_plan_echo_imrt
0,Lung_Phantom_Patient_1,data/Lung_Phantom_Patient_1/CT_Data.h5,data/Lung_Phantom_Patient_1/CT_MetaData.json,data/Lung_Phantom_Patient_1/StructureSet_Data.h5,data/Lung_Phantom_Patient_1/StructureSet_MetaD...,[data/Lung_Phantom_Patient_1/Beams/Beam_0_Data...,[data/Lung_Phantom_Patient_1/Beams/Beam_0_Meta...,data/Lung_Phantom_Patient_1/OptimizationVoxels...,data/Lung_Phantom_Patient_1/OptimizationVoxels...,data/Lung_Phantom_Patient_1/PlannerBeams.json,NaN,NaN
1,Lung_Patient_2,data/Lung_Patient_2/CT_Data.h5,data/Lung_Patient_2/CT_MetaData.json,data/Lung_Patient_2/StructureSet_Data.h5,data/Lung_Patient_2/StructureSet_MetaData.json,"[data/Lung_Patient_2/Beams/Beam_0_Data.h5, dat...",[data/Lung_Patient_2/Beams/Beam_0_MetaData.jso...,data/Lung_Patient_2/OptimizationVoxels_Data.h5,data/Lung_Patient_2/OptimizationVoxels_MetaDat...,data/Lung_Patient_2/PlannerBeams.json,data/Lung_Patient_2/DicomFiles/rt_dose_echo_im...,data/Lung_Patient_2/DicomFiles/rt_plan_echo_im...
2,Lung_Patient_3,data/Lung_Patient_3/CT_Data.h5,data/Lung_Patient_3/CT_MetaData.json,data/Lung_Patient_3/StructureSet_Data.h5,data/Lung_Patient_3/StructureSet_MetaData.json,"[data/Lung_Patient_3/Beams/Beam_0_Data.h5, dat...",[data/Lung_Patient_3/Beams/Beam_0_MetaData.jso...,data/Lung_Patient_3/OptimizationVoxels_Data.h5,data/Lung_Patient_3/OptimizationVoxels_MetaDat...,data/Lung_Patient_3/PlannerBeams.json,data/Lung_Patient_3/DicomFiles/rt_dose_echo_im...,data/Lung_Patient_3/DicomFiles/rt_plan_echo_im...
3,Lung_Patient_4,data/Lung_Patient_4/CT_Data.h5,data/Lung_Patient_4/CT_MetaData.json,data/Lung_Patient_4/StructureSet_Data.h5,data/Lung_Patient_4/StructureSet_MetaData.json,"[data/Lung_Patient_4/Beams/Beam_0_Data.h5, dat...",[data/Lung_Patient_4/Beams/Beam_0_MetaData.jso...,data/Lung_Patient_4/OptimizationVoxels_Data.h5,data/Lung_Patient_4/OptimizationVoxels_MetaDat...,data/Lung_Patient_4/PlannerBeams.json,data/Lung_Patient_4/DicomFiles/rt_dose_echo_im...,data/Lung_Patient_4/DicomFiles/rt_plan_echo_im...
4,Lung_Patient_5,data/Lung_Patient_5/CT_Data.h5,data/Lung_Patient_5/CT_MetaData.json,data/Lung_Patient_5/StructureSet_Data.h5,data/Lung_Patient_5/StructureSet_MetaData.json,"[data/Lung_Patient_5/Beams/Beam_0_Data.h5, dat...",[data/Lung_Patient_5/Beams/Beam_0_MetaData.jso...,data/Lung_Patient_5/OptimizationVoxels_Data.h5,data/Lung_Patient_5/OptimizationVoxels_MetaDat...,data/Lung_Patient_5/PlannerBeams.json,data/Lung_Patient_5/DicomFiles/rt_dose_echo_im...,data/Lung_Patient_5/DicomFiles/rt_plan_echo_im...


In [ ]:
# Run this first. Restart kernel after this cell completes.
!pip install portpy cvxpy pymoo scipy matplotlib numpy h5py

In [ ]:
import portpy.photon as pp
import cvxpy as cp
import numpy as np
import scipy
import pymoo
import matplotlib.pyplot as plt

print("portpy   :", pp.__version__ if hasattr(pp, '__version__') else "OK")
print("cvxpy    :", cp.__version__)
print("numpy    :", np.__version__)
print("pymoo    :", pymoo.__version__)
print("All imports successful ✓")

portpy   : OK
cvxpy    : 1.6.7
numpy    : 1.26.4
pymoo    : 0.6.1.6
All imports successful ✓


In [ ]:
# This downloads the dataset from HuggingFace on first run (~2-5 min)
# On subsequent runs it loads from local cache instantly
data = pp.DataExplorer(
    hf_repo_id="PortPy-Project/PortPy_Dataset",
    local_download_dir='./data'
)

# ✅ Correct: assign patient_id as attribute, NOT data.set_patient()
#data.patient_id = 'Lung_Patient_3'

#print("Patient selected:", data.patient_id)

In [ ]:
import portpy.photon as pp

# The dataset downloads into a subfolder called 'data/' inside your download dir
# So set local_download_dir one level up to avoid the double 'data/data/' issue
data = pp.DataExplorer(
    hf_repo_id="PortPy-Project/PortPy_Dataset",
    local_download_dir='.'    # <-- changed from './data' to '.'
)

# Files are already downloaded so this will just point to ./data/Lung_Patient_3
data.patient_id = 'Lung_Patient_3'
print("Patient selected:", data.patient_id)

Patient selected: Lung_Patient_3


In [ ]:
data.patient_id


'Lung_Patient_3'

In [ ]:
import os

# Search for any StructureSet file anywhere under current directory
for root, dirs, files in os.walk('.'):
    for f in files:
        if 'StructureSet' in f:
            print(os.path.join(root, f))
"""

Run that and share the output. It will print the exact path like:
```
./data/data/Lung_Patient_3/StructureSet_Data.h5
```
or
```
./PortPy_Dataset/data/Lung_Patient_3/StructureSet_Data.h5"""

'\n\nRun that and share the output. It will print the exact path like:\n```\n./data/data/Lung_Patient_3/StructureSet_Data.h5\n```\nor\n```\n./PortPy_Dataset/data/Lung_Patient_3/StructureSet_Data.h5'

In [ ]:
# Look at what's available for Lung_Patient_3
import pandas as pd

# Find the row for Lung_Patient_3
df = ds['train'].to_pandas()
patient_row = df[df['patient_id'] == 'Lung_Patient_3'].iloc[0]
print(patient_row.index.tolist())  # print all column names
print("\nct_data type      :", type(patient_row['ct_data']))
print("structureset type :", type(patient_row['structureset_data']))
print("beam_data type    :", type(patient_row['beam_data_paths']))

['patient_id', 'ct_data', 'ct_metadata', 'structureset_data', 'structureset_metadata', 'beam_data_paths', 'beam_metadata_paths', 'optimization_voxels_data', 'optimization_voxels_metadata', 'planner_beams', 'rt_dose_echo_imrt', 'rt_plan_echo_imrt']

ct_data type      : <class 'str'>
structureset type : <class 'str'>
beam_data type    : <class 'numpy.ndarray'>


In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import os

# Download just Lung_Patient_3 (not all 229 patients)
# This downloads all files for that one patient into ./hf_data/
snapshot_download(
    repo_id="PortPy-Project/PortPy_Dataset",
    repo_type="dataset",
    allow_patterns="data/Lung_Patient_3/*",   # only this patient
    local_dir="./hf_data"
)

print("Download complete ✓")
print("Files downloaded to ./hf_data/data/Lung_Patient_3/")

Fetching ... files: 0it [00:00, ?it/s]

In [ ]:
base = './hf_data/data/Lung_Patient_3'

for f in os.listdir(base):
    print(f)

In [ ]:
import portpy.photon as pp

data = pp.DataExplorer(data_dir='./hf_data/data')
data.patient_id = 'Lung_Patient_3'

print("DataExplorer ready ✓")
import h5py

struct_path = './hf_data/data/Lung_Patient_3/StructureSet_Data.h5'
with h5py.File(struct_path, 'r') as f:
    print("Available structures:")
    def print_keys(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(" ", name)
    f.visititems(print_keys)


In [ ]:
import inspect
print(inspect.signature(pp.InfluenceMatrix.__init__))

In [ ]:
# Try this order first (beams before structs)
try:
    inf_matrix = pp.InfluenceMatrix(ct=ct,beams=beams,structs=structs)
    print("Option 1 worked ✓")
except Exception as e:
    print(f"Option 1 failed: {e}")

In [ ]:
import os
import h5py
import numpy as np
from scipy.sparse import csr_matrix, vstack

beam_dir = './hf_data/data/Lung_Patient_3/Beams'
beam_files = sorted([
    os.path.join(beam_dir, f)
    for f in os.listdir(beam_dir)
    if f.endswith('_Data.h5')
])

print(f"Found {len(beam_files)} beam data files")
print("First beam file keys:")

# Inspect first beam to understand structure
with h5py.File(beam_files[0], 'r') as f:
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
    f.visititems(show)

In [ ]:
with h5py.File(beam_files[0], 'r') as f:
    sparse_ds = f['inf_matrix_sparse']
    full_ds   = f['inf_matrix_full']

    print(f"inf_matrix_sparse : shape={sparse_ds.shape}, dtype={sparse_ds.dtype}")
    print(f"inf_matrix_full   : shape={full_ds.shape},   dtype={full_ds.dtype}")

    # Peek at first few values
    print(f"\nFirst row of sparse:\n{sparse_ds[0]}")
    print(f"\nFirst row of full:\n{full_ds[0][:10]}...")  # first 10 values

In [ ]:
from scipy.sparse import coo_matrix, hstack

A_blocks = []
n_voxels_per_beam = []

with h5py.File(beam_files[0], 'r') as f:
    n_voxels = f['inf_matrix_full'].shape[0]
print(f"Consistent n_voxels: {n_voxels}")

for bf in sorted(beam_files):
    with h5py.File(bf, 'r') as f:
        coo_data = f['inf_matrix_sparse'][:]   # shape: (nnz, 3)
        n_cols   = f['inf_matrix_full'].shape[1]   # beamlets for this beam

        rows = coo_data[:, 0].astype(np.int32)
        cols = coo_data[:, 1].astype(np.int32)
        vals = coo_data[:, 2].astype(np.float32)

        A_beam = coo_matrix((vals, (rows, cols)),
                            shape=(n_voxels, n_cols)).tocsr()
        A_blocks.append(A_beam)

A = hstack(A_blocks, format='csr')
print(f"A matrix shape : {A.shape}")
print(f"Num voxels     : {A.shape[0]}")
print(f"Num beamlets   : {A.shape[1]}")
print(f"Memory (MB)    : {A.data.nbytes / 1e6:.1f}")

In [ ]:
with h5py.File('./hf_data/data/Lung_Patient_3/StructureSet_Data.h5', 'r') as f:
    print("StructureSet full tree:")
    def show_all(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
    f.visititems(show_all)

In [ ]:
with h5py.File('./hf_data/data/Lung_Patient_3/OptimizationVoxels_Data.h5', 'r') as f:
    print("OptimizationVoxels tree:")
    def show_all(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
    f.visititems(show_all)

In [ ]:
with h5py.File('./hf_data/data/Lung_Patient_3/OptimizationVoxels_Data.h5', 'r') as f:
    ct_to_dose = f['ct_to_dose_voxel_map'][:]   # shape: (107, 512, 512)

print(f"Shape          : {ct_to_dose.shape}")
print(f"Dtype          : {ct_to_dose.dtype}")
print(f"Min value      : {ct_to_dose.min()}")
print(f"Max value      : {ct_to_dose.max()}")
print(f"Unique values  : {len(np.unique(ct_to_dose))}")
print(f"How many == -1 : {(ct_to_dose == -1).sum():,}")   # voxels NOT in opt grid
print(f"How many >= 0  : {(ct_to_dose >= 0).sum():,}")    # voxels IN opt grid
print(f"A matrix rows  : {A.shape[0]}")                   # should match >= 0 count

In [ ]:
# ct_to_dose_voxel_map tells you:
# For each CT voxel at position (z, y, x):
#   -1  → this voxel is NOT in the optimization grid (skip it)
#   k   → this voxel IS row k in the A matrix

# Example: what A-matrix row does CT voxel (50, 256, 256) map to?
z, y, x = 50, 256, 256
row_in_A = ct_to_dose[z, y, x]
print(f"CT voxel ({z},{y},{x}) → A matrix row {row_in_A}")

# Visualise one CT slice of the map
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: which voxels are in the opt grid (white = yes, black = no)
axes[0].imshow(ct_to_dose[50] >= 0, cmap='gray')
axes[0].set_title('Optimization voxels (slice 50)\nWhite = included in A matrix')

# Right: the actual row index values
im = axes[1].imshow(np.where(ct_to_dose[50] >= 0, ct_to_dose[50], np.nan),
                    cmap='viridis')
axes[1].set_title('A matrix row index per voxel (slice 50)')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.savefig('opt_voxel_map.png', dpi=100)
plt.show()

In [ ]:
with h5py.File('./hf_data/data/Lung_Patient_3/StructureSet_Data.h5', 'r') as f:
    mask_ptv   = f['PTV'][:]        # shape: (107, 512, 512), binary
    mask_esoph = f['ESOPHAGUS'][:]
    mask_cord  = f['CORD'][:]
    mask_lung  = f['LUNG_L'][:]
    mask_heart = f['HEART'][:]

def get_voxel_indices(mask_3d, ct_to_dose_map):
    """
    mask_3d       : binary (107, 512, 512) — 1 where structure exists
    ct_to_dose_map: int32  (107, 512, 512) — A matrix row index, -1 if not in opt grid
    Returns: 1D array of row indices into A matrix
    """
    # Voxels that are both IN the structure AND IN the opt grid
    in_structure   = mask_3d.astype(bool)
    in_opt_grid    = ct_to_dose_map >= 0
    both           = in_structure & in_opt_grid

    # Get their A matrix row indices
    row_indices    = ct_to_dose_map[both]
    return row_indices.astype(np.int32)

ptv_voxels   = get_voxel_indices(mask_ptv,   ct_to_dose)
esoph_voxels = get_voxel_indices(mask_esoph, ct_to_dose)
cord_voxels  = get_voxel_indices(mask_cord,  ct_to_dose)
lung_voxels  = get_voxel_indices(mask_lung,  ct_to_dose)
heart_voxels = get_voxel_indices(mask_heart, ct_to_dose)

print(f"PTV       : {len(ptv_voxels):,} voxels in opt grid")
print(f"Esophagus : {len(esoph_voxels):,} voxels in opt grid")
print(f"Cord      : {len(cord_voxels):,} voxels in opt grid")
print(f"Lung_L    : {len(lung_voxels):,} voxels in opt grid")
print(f"Heart     : {len(heart_voxels):,} voxels in opt grid")

In [ ]:
A_ptv   = A[ptv_voxels, :]
A_esoph = A[esoph_voxels, :]
A_cord  = A[cord_voxels, :]
A_lung  = A[lung_voxels, :]

print(f"A_ptv   : {A_ptv.shape}")
print(f"A_esoph : {A_esoph.shape}")
print(f"A_cord  : {A_cord.shape}")
print(f"A_lung  : {A_lung.shape}")

# Sanity check with unit intensity beams
x_test     = np.ones(A.shape[1], dtype=np.float32)
dose_ptv   = A_ptv   @ x_test
dose_esoph = A_esoph @ x_test
dose_cord  = A_cord  @ x_test

print(f"\nWith unit beam intensities:")
print(f"  Mean PTV dose       : {dose_ptv.mean():.4f}")
print(f"  Mean Esophagus dose : {dose_esoph.mean():.4f}")
print(f"  Mean Cord dose      : {dose_cord.mean():.4f}")

In [ ]:
import json

with open('./hf_data/data/Lung_Patient_3/PlannerBeams.json') as f:
    planner = json.load(f)

print("PlannerBeams.json content:")
print(json.dumps(planner, indent=2))

In [ ]:
# planner will look something like:
# {"beam_ids": [4, 15, 27, 39, 51, 63]}  or  [4, 15, 27, 39, 51, 63]
# Adjust based on what you see printed above

# Extract beam IDs
if isinstance(planner, dict):
    beam_ids = planner.get('beam_ids', planner.get('BeamId', list(planner.values())[0]))
else:
    beam_ids = planner  # it's already a list

print(f"Planner selected {len(beam_ids)} beams: {beam_ids}")

# Each beam contributes a block of columns to A
# We need to know how many beamlets each beam has
beamlets_per_beam = []
for bf in sorted(beam_files):
    with h5py.File(bf, 'r') as f:
        n_beamlets_this_beam = f['inf_matrix_full'].shape[1]
        beamlets_per_beam.append(n_beamlets_this_beam)

print(f"\nBeamlets per beam (first 5): {beamlets_per_beam[:5]}")
print(f"Total beamlets: {sum(beamlets_per_beam)} (should match {A.shape[1]})")

# Build column index ranges for each beam
beam_col_start = np.cumsum([0] + beamlets_per_beam)
print(f"\nBeam 0 columns : {beam_col_start[0]} to {beam_col_start[1]-1}")
print(f"Beam 1 columns : {beam_col_start[1]} to {beam_col_start[2]-1}")

# Select only columns belonging to planner beams
planner_col_indices = np.concatenate([
    np.arange(beam_col_start[b], beam_col_start[b+1])
    for b in beam_ids
])

print(f"\nTotal planner beamlets: {len(planner_col_indices)}")

# Subset A matrix to planner beams only
A_plan     = A[:, planner_col_indices]
A_ptv_plan   = A_ptv[:,   planner_col_indices]
A_esoph_plan = A_esoph[:, planner_col_indices]
A_cord_plan  = A_cord[:,  planner_col_indices]
A_lung_plan  = A_lung[:,  planner_col_indices]

print(f"\nReduced A matrix shape : {A_plan.shape}")
print(f"A_ptv_plan shape       : {A_ptv_plan.shape}")

In [ ]:
x_test_plan = np.ones(A_plan.shape[1], dtype=np.float32)

print("Sanity check on planner-beam-only A matrix:")
print(f"  Mean PTV dose       : {(A_ptv_plan   @ x_test_plan).mean():.4f} Gy")
print(f"  Mean Esophagus dose : {(A_esoph_plan @ x_test_plan).mean():.4f} Gy")
print(f"  Mean Cord dose      : {(A_cord_plan  @ x_test_plan).mean():.4f} Gy")
print(f"\nReady for optimization ✓")
print(f"Problem size: {A_plan.shape[1]} beamlet variables, "
      f"{len(ptv_voxels)+len(esoph_voxels)+len(cord_voxels):,} constrained voxels")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load CT for background
with h5py.File('./hf_data/data/Lung_Patient_3/CT_Data.h5', 'r') as f:
    ct_keys = list(f.keys())
    print("CT keys:", ct_keys)
    ct_hu = f[ct_keys[0]][:]   # grab first dataset

# Pick a representative slice (middle of PTV)
# Find which slice has most PTV voxels
ptv_per_slice = mask_ptv.sum(axis=(1,2))
best_slice    = np.argmax(ptv_per_slice)
print(f"Best slice to visualise: {best_slice} "
      f"(has {ptv_per_slice[best_slice]} PTV voxels)")

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(ct_hu[best_slice], cmap='gray', vmin=-1000, vmax=400)

# Overlay structures as contours
colours = {
    'PTV'      : ('red',    mask_ptv),
    'ESOPHAGUS': ('yellow', mask_esoph),
    'CORD'     : ('blue',   mask_cord),
    'LUNG_L'   : ('green',  mask_lung),
}
for name, (colour, mask) in colours.items():
    ax.contour(mask[best_slice], levels=[0.5], colors=colour, linewidths=1.5)
    ax.plot([], [], color=colour, label=name)   # legend proxy

ax.legend(loc='upper right', fontsize=10)
ax.set_title(f'CT Slice {best_slice} with Structure Contours', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig('ct_structures.png', dpi=150)
plt.show()

In [ ]:
# The optimizer needs to find x such that A_ptv_plan @ x ≈ 60 Gy
# Let's understand the required scale

# If all beamlets had equal intensity k, what k achieves mean PTV dose = 60?
mean_dose_per_unit = (A_ptv_plan @ np.ones(4420)).mean()
k_required = 60.0 / mean_dose_per_unit
print(f"Mean PTV dose per unit intensity : {mean_dose_per_unit:.4f} Gy")
print(f"Uniform intensity to hit 60 Gy   : {k_required:.4f}")
print(f"So x values will be around       : {k_required:.2f} MU (monitor units)")

# Check what that uniform plan does to OARs
x_uniform = np.full(4420, k_required, dtype=np.float32)
d_ptv_unif   = A_ptv_plan   @ x_uniform
d_esoph_unif = A_esoph_plan @ x_uniform
d_cord_unif  = A_cord_plan  @ x_uniform
d_lung_unif  = A_lung_plan  @ x_uniform

print(f"\nUniform plan (x = {k_required:.2f} everywhere):")
print(f"  PTV   — mean: {d_ptv_unif.mean():.1f} Gy, "
      f"D95: {np.percentile(d_ptv_unif, 5):.1f} Gy")    # D95 = dose to 95% of volume
print(f"  Esoph — mean: {d_esoph_unif.mean():.1f} Gy, "
      f"max: {d_esoph_unif.max():.1f} Gy  (limit: 34 Gy)")
print(f"  Cord  — mean: {d_cord_unif.mean():.1f} Gy,  "
      f"max: {d_cord_unif.max():.1f} Gy  (limit: 45 Gy)")
print(f"  Lung  — mean: {d_lung_unif.mean():.1f} Gy  (limit: 20 Gy)")

In [ ]:
print("=" * 55)
print("  PROBLEM SUMMARY — READY FOR OPTIMIZATION")
print("=" * 55)
print(f"  Patient          : Lung_Patient_3")
print(f"  Beams selected   : 7  (IDs: 0,6,12,18,24,30,35)")
print(f"  Beamlets (vars)  : {A_plan.shape[1]:,}  ← size of x")
print(f"  Opt voxels       : {A_plan.shape[0]:,}")
print()
print(f"  OBJECTIVES:")
print(f"    F1 — PTV underdose   : {len(ptv_voxels):,} voxels, target ≥ 60 Gy")
print(f"    F2 — OAR overdose    :")
print(f"         Esophagus        : {len(esoph_voxels):,} voxels, limit 34 Gy")
print(f"         Cord             : {len(cord_voxels):,} voxels, limit 45 Gy")
print(f"         Lung mean        : {len(lung_voxels):,} voxels, limit 20 Gy")
print(f"    F3 — Spectral reg    : nuclear norm of reshaped x")
print()
print(f"  SOLVERS TO RUN:")
print(f"    1. CVXPY (convex, weighted sum) → baseline")
print(f"    2. CVXPY + spectral reg         → your contribution")
print(f"    3. NSGA-II (evolutionary)       → comparison")
print(f"    4. Pareto front + DVH comparison")
print("=" * 55)

In [ ]:
A_plan_sparse = csr_matrix(A_plan)
#print first element in A
print(A_plan_sparse[0])

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
import time
import warnings
warnings.filterwarnings('ignore')

# Confirm all matrices are ready
print("Problem dimensions:")
print(f"  Beamlets (x size)  : {A_plan.shape[1]}")
print(f"  PTV voxels         : {A_ptv_plan.shape[0]:,}")
print(f"  Esophagus voxels   : {A_esoph_plan.shape[0]:,}")
print(f"  Cord voxels        : {A_cord_plan.shape[0]:,}")
print(f"  Lung voxels        : {A_lung_plan.shape[0]:,}")

# Clinical dose limits
D_PTV   = 60.0   # Gy — prescription (must deliver this)
D_ESOPH = 34.0   # Gy — esophagus tolerance
D_CORD  = 45.0   # Gy — spinal cord tolerance
D_LUNG  = 20.0   # Gy — mean lung dose tolerance

n_beamlets = A_plan.shape[1]   # 4420

In [ ]:
import h5py
import numpy as np

# Look inside the actual file that A comes from
beam_path = '/content/hf_data/data/Lung_Patient_3/Beams/Beam_0_Data.h5'

with h5py.File(beam_path, 'r') as f:

    # The raw sparse storage — shape (N, 3)
    raw = f['inf_matrix_sparse'][:]

    print("=" * 60)
    print("FILE: Beam_0_Data.h5  →  inf_matrix_sparse")
    print("=" * 60)
    print(f"Shape  : {raw.shape}")
    print(f"dtype  : {raw.dtype}")
    print()
    print("What each column means:")
    print("  col 0 = voxel row index  (which voxel receives this dose)")
    print("  col 1 = beamlet col index (which sub-beam delivers it)")
    print("  col 2 = dose value in Gy  (at unit beam intensity)")
    print()
    print("First 5 nonzero entries:")
    print(f"  {'voxel':>8}  {'beamlet':>8}  {'dose(Gy)':>10}")
    print(f"  {'-'*8}  {'-'*8}  {'-'*10}")
    for row in raw[:5]:
        print(f"  {int(row[0]):>8}  {int(row[1]):>8}  {row[2]:>10.6f}")

    print()
    print(f"Total nonzero entries : {len(raw):,}")
    print(f"Max voxel index       : {int(raw[:,0].max()):,}  (out of 432,753)")
    print(f"Max beamlet index     : {int(raw[:,1].max()):,}")
    print(f"Max dose value        : {raw[:,2].max():.6f} Gy")
    print(f"Min dose value (>0)   : {raw[raw[:,2]>0, 2].min():.6f} Gy")

In [ ]:
print("=" * 60)
print("ASSEMBLED A MATRIX  (all 72 beams combined)")
print("=" * 60)
print(f"Shape   : {A.shape}")
print(f"         ↑       ↑")
print(f"    432,753   46,733")
print(f"    voxels    beamlets")
print()
print(f"Type    : {type(A)}")
print(f"Format  : CSR sparse (Compressed Sparse Row)")
print(f"Memory  : {A.data.nbytes / 1e6:.1f} MB")
print(f"Nonzeros: {A.nnz:,}")
print(f"Density : {100 * A.nnz / (A.shape[0]*A.shape[1]):.4f}% filled")
print()

# Print 5 actual nonzero values from A
cx = A.tocoo()   # convert to COO to easily access row, col, data
print("5 sample nonzero entries from A:")
print(f"  {'voxel(row)':>12}  {'beamlet(col)':>14}  {'dose(Gy)':>10}")
print(f"  {'-'*12}  {'-'*14}  {'-'*10}")
for i in range(5):
    print(f"  {cx.row[i]:>12}  {cx.col[i]:>14}  {cx.data[i]:>10.6f}")

print()
print("Interpretation of first entry:")
r, c, v = cx.row[0], cx.col[0], cx.data[0]
print(f"  Voxel {r} receives {v:.6f} Gy from beamlet {c} at intensity 1.0")

In [ ]:
print("=" * 60)
print("A_plan  (7 planner beams only — used in optimization)")
print("=" * 60)
print(f"Shape   : {A_plan.shape}")
print(f"         ↑      ↑")
print(f"    432,753   4,420")
print(f"    voxels    beamlets (7 beams × ~631 each)")
print()

cx_plan = A_plan.tocoo()
print("5 sample entries from A_plan:")
print(f"  {'voxel':>8}  {'beamlet':>8}  {'dose(Gy)':>10}")
print(f"  {'-'*8}  {'-'*8}  {'-'*10}")
for i in range(5):
    print(f"  {cx_plan.row[i]:>8}  {cx_plan.col[i]:>8}  {cx_plan.data[i]:>10.6f}")

In [ ]:
print("=" * 60)
print("STRUCTURE SUB-MATRICES")
print("(rows of A_plan selected by structure voxel indices)")
print("=" * 60)

for name, mat, voxels in [
    ("A_ptv_plan",   A_ptv_plan,   ptv_voxels),
    ("A_esoph_plan", A_esoph_plan, esoph_voxels),
    ("A_cord_plan",  A_cord_plan,  cord_voxels),
]:
    cx_s = mat.tocoo()
    print(f"\n{name}:")
    print(f"  Shape : {mat.shape}  ← ({mat.shape[0]} structure voxels × 4420 beamlets)")
    print(f"  These are rows {voxels[:3]} ... of A_plan")
    print(f"  5 sample entries:")
    print(f"    {'struct_voxel':>14}  {'beamlet':>8}  {'dose(Gy)':>10}")
    print(f"    {'-'*14}  {'-'*8}  {'-'*10}")
    for i in range(min(5, len(cx_s.data))):
        print(f"    {cx_s.row[i]:>14}  {cx_s.col[i]:>8}  {cx_s.data[i]:>10.6f}")

    # Show what unit beams give
    d_unit = mat @ np.ones(mat.shape[1], dtype=np.float32)
    print(f"  Dose stats at unit intensity (x = all 1s):")
    print(f"    mean={d_unit.mean():.4f}  min={d_unit.min():.4f}  max={d_unit.max():.4f} Gy")

## Optimization Setup — Sparse Matrices (no dense conversion)

We keep everything in sparse format. `todense()` would allocate ~1 GB and crash Colab.
Instead we work directly on `.data` (the nonzero values only).

In [ ]:
import numpy as np
import scipy.sparse as sp
import scs
import time
import warnings
warnings.filterwarnings('ignore')

# Clinical dose limits (Gy)
D_PTV   = 60.0
D_ESOPH = 34.0
D_CORD  = 45.0
D_LUNG  = 20.0

# Cast to float64 sparse — NO .todense() call
A_ptv_sp   = A_ptv_plan.astype(np.float64)
A_esoph_sp = A_esoph_plan.astype(np.float64)
A_cord_sp  = A_cord_plan.astype(np.float64)
A_lung_sp  = A_lung_plan.astype(np.float64)

print('Sparse matrices ready (no dense conversion)')
for name, m in [('A_ptv', A_ptv_sp), ('A_esoph', A_esoph_sp),
                ('A_cord', A_cord_sp), ('A_lung', A_lung_sp)]:
    mb = (m.data.nbytes + m.indices.nbytes + m.indptr.nbytes) / 1e6
    print(f'  {name:<10} shape={m.shape}  sparse size={mb:.1f} MB')


## Solve WITHOUT Spectral Regularization

**Why direct SCS and not CVXPY?**  
CVXPY internally converts problems to COO format and allocates dense float64 arrays during canonicalization — this causes a MemoryError on large problems like ours.  
Calling `scs.SCS()` directly accepts CSC sparse matrices and stays sparse throughout.

**Linearisation of max(v, 0):**  
We introduce slack variables t and write:  
`min sum(t)  s.t.  t >= violation,  t >= 0`  
This converts the nonsmooth hinge loss into a standard linear cone program.

In [ ]:
def solve_no_spectral(A_ptv, A_esoph, A_cord, A_lung, w1=1.0, w2=2.0):
    t0 = time.time()
    n  = A_ptv.shape[1]    # 4420 beamlets  <- what we solve for
    nP = A_ptv.shape[0]    # 58845 PTV voxels
    nE = A_esoph.shape[0]  # 10885 esoph voxels
    nC = A_cord.shape[0]   # 7649  cord voxels
    nL = A_lung.shape[0]   # 334755 lung voxels

    # Variable layout: z = [x(n), t_ptv(nP), t_esoph(nE), t_cord(nC)]
    n_vars = n + nP + nE + nC
    print(f'  Variables: {n_vars:,}')

    # Objective: x has zero cost, slacks have cost = weight/n_voxels
    c = np.zeros(n_vars)
    c[n:n+nP]       = w1 / nP  # PTV underdose cost
    c[n+nP:n+nP+nE] = w2 / nE  # esoph overdose cost
    c[n+nP+nE:]     = w2 / nC  # cord overdose cost

    rows_ub = []; rhs_ub = []

    # (1) PTV underdose: t_ptv >= D_PTV - A_ptv*x  =>  -A_ptv*x - t_ptv <= -D_PTV
    A1 = sp.hstack([-A_ptv, -sp.eye(nP),
                    sp.csr_matrix((nP, nE)), sp.csr_matrix((nP, nC))], format='csr')
    rows_ub.append(A1); rhs_ub.append(-D_PTV * np.ones(nP))

    # (2) Esoph overdose: t_esoph >= A_esoph*x - D_ESOPH  =>  A_esoph*x - t_esoph <= D_ESOPH
    A2 = sp.hstack([A_esoph, sp.csr_matrix((nE, nP)),
                    -sp.eye(nE), sp.csr_matrix((nE, nC))], format='csr')
    rows_ub.append(A2); rhs_ub.append(D_ESOPH * np.ones(nE))

    # (3) Cord overdose: A_cord*x - t_cord <= D_CORD
    A3 = sp.hstack([A_cord, sp.csr_matrix((nC, nP)),
                    sp.csr_matrix((nC, nE)), -sp.eye(nC)], format='csr')
    rows_ub.append(A3); rhs_ub.append(D_CORD * np.ones(nC))

    # (4) Hard lung mean constraint: (1/nL) * 1^T * A_lung * x <= D_LUNG
    lung_row = (A_lung.T @ np.ones(nL) / nL).reshape(1, -1)
    A4 = sp.hstack([lung_row, sp.csr_matrix((1, nP + nE + nC))], format='csr')
    rows_ub.append(A4); rhs_ub.append(np.array([D_LUNG]))

    # (5) Non-negativity of all variables: -z <= 0
    rows_ub.append(-sp.eye(n_vars, format='csr'))
    rhs_ub.append(np.zeros(n_vars))

    A_ub = sp.vstack(rows_ub, format='csc')
    b_ub = np.concatenate(rhs_ub)
    n_ineq = A_ub.shape[0]
    print(f'  Constraints: {n_ineq:,}  |  A_ub nnz: {A_ub.nnz:,}')

    # Call SCS directly — fully sparse, no CVXPY canonicalization
    data   = {'c': c, 'A': A_ub, 'b': b_ub}
    cone   = {'l': n_ineq}
    solver = scs.SCS(data, cone, eps=1e-4, max_iters=5000, verbose=False)
    sol    = solver.solve()

    elapsed = time.time() - t0
    xv = sol['x']
    x_opt = xv[:n] if xv is not None else None
    F1 = float(np.mean(np.maximum(D_PTV - A_ptv @ x_opt, 0))) if x_opt is not None else np.nan
    F2 = float(np.mean(np.maximum(A_esoph @ x_opt - D_ESOPH, 0))
             + np.mean(np.maximum(A_cord  @ x_opt - D_CORD, 0))) if x_opt is not None else np.nan
    return {'x': x_opt, 'status': sol['info']['status'],
            'F1': F1, 'F2': F2, 'F_spectral': 0.0,
            'solve_time': elapsed, 'label': 'No Spectral Reg'}

print('Solving WITHOUT spectral regularization...')
r_no_spec = solve_no_spectral(A_ptv_sp, A_esoph_sp, A_cord_sp, A_lung_sp)
print(f"  Status : {r_no_spec['status']}")
print(f"  F1 PTV : {r_no_spec['F1']:.4f} Gy/voxel")
print(f"  F2 OAR : {r_no_spec['F2']:.4f} Gy/voxel")
print(f"  Time   : {r_no_spec['solve_time']:.1f} s")


## Solve WITH Spectral (Nuclear Norm) Regularization

**What is nuclear norm regularization?**  
We reshape x into a matrix X of shape (n_beams × beamlets_per_beam).  
Nuclear norm = sum of singular values of X.  
Minimising this forces beam rows to be correlated (low rank)  
= each beam delivers a similar pattern = physically smooth fluence maps.

**SDP reformulation:**  
||X||_* = min  (trace(P) + trace(Q)) / 2  
s.t.  [[P, X], [X^T, Q]] is positive semidefinite  
SCS handles PSD constraints natively via its 's' cone.

In [ ]:
def solve_with_spectral(A_ptv, A_esoph, A_cord, A_lung,
                        n_beams, bpb, w1=1.0, w2=2.0, lam=0.05):
    t0 = time.time()
    n  = A_ptv.shape[1]; nP = A_ptv.shape[0]
    nE = A_esoph.shape[0]; nC = A_cord.shape[0]; nL = A_lung.shape[0]
    nb      = n_beams
    min_bpb = min(bpb)        # use minimum beamlets across beams for uniform matrix
    n_unif  = nb * min_bpb    # beamlets used in nuclear norm matrix X
    nP_sdp  = nb * nb         # entries in P (nb x nb PSD matrix)
    nQ_sdp  = min_bpb * min_bpb  # entries in Q

    # Variable layout: [x, t_ptv, t_esoph, t_cord, t_nuc, P_vec, Q_vec]
    n_vars  = n + nP + nE + nC + 1 + nP_sdp + nQ_sdp
    ix_tp   = n
    ix_te   = n + nP
    ix_tc   = n + nP + nE
    ix_tnuc = n + nP + nE + nC
    ix_P    = ix_tnuc + 1
    ix_Q    = ix_P + nP_sdp
    print(f'  n_vars={n_vars:,}  X shape=({nb}x{min_bpb})')

    c = np.zeros(n_vars)
    c[ix_tp:ix_tp+nP]   = w1 / nP
    c[ix_te:ix_te+nE]   = w2 / nE
    c[ix_tc:ix_tc+nC]   = w2 / nC
    c[ix_tnuc]          = lam

    rows_ub = []; rhs_ub = []
    pad1 = sp.csr_matrix((nP, 1 + nP_sdp + nQ_sdp))
    A1 = sp.hstack([-A_ptv, -sp.eye(nP), sp.csr_matrix((nP, nE)),
                    sp.csr_matrix((nP, nC)), pad1], format='csr')
    rows_ub.append(A1); rhs_ub.append(-D_PTV * np.ones(nP))

    pad2 = sp.csr_matrix((nE, 1 + nP_sdp + nQ_sdp))
    A2 = sp.hstack([A_esoph, sp.csr_matrix((nE, nP)),
                    -sp.eye(nE), sp.csr_matrix((nE, nC)), pad2], format='csr')
    rows_ub.append(A2); rhs_ub.append(D_ESOPH * np.ones(nE))

    pad3 = sp.csr_matrix((nC, 1 + nP_sdp + nQ_sdp))
    A3 = sp.hstack([A_cord, sp.csr_matrix((nC, nP)), sp.csr_matrix((nC, nE)),
                    -sp.eye(nC), pad3], format='csr')
    rows_ub.append(A3); rhs_ub.append(D_CORD * np.ones(nC))

    lung_row = (A_lung.T @ np.ones(nL) / nL).reshape(1, -1)
    A4 = sp.hstack([lung_row,
                    sp.csr_matrix((1, nP + nE + nC + 1 + nP_sdp + nQ_sdp))], format='csr')
    rows_ub.append(A4); rhs_ub.append(np.array([D_LUNG]))

    n_nn = n + nP + nE + nC + 1
    A5 = sp.hstack([-sp.eye(n_nn),
                    sp.csr_matrix((n_nn, nP_sdp + nQ_sdp))], format='csr')
    rows_ub.append(A5); rhs_ub.append(np.zeros(n_nn))

    # Nuclear norm trace constraint: -t_nuc + 0.5*trace(P) + 0.5*trace(Q) <= 0
    tP = np.zeros(nP_sdp); tQ = np.zeros(nQ_sdp)
    for i in range(nb):      tP[i*nb + i]          = 0.5
    for i in range(min_bpb): tQ[i*min_bpb + i]     = 0.5
    A6 = sp.hstack([sp.csr_matrix((1, n + nP + nE + nC)),
                    sp.csr_matrix([[-1.0]]),
                    sp.csr_matrix(tP.reshape(1, -1)),
                    sp.csr_matrix(tQ.reshape(1, -1))], format='csr')
    rows_ub.append(A6); rhs_ub.append(np.array([0.0]))

    A_ub = sp.vstack(rows_ub, format='csc')
    b_ub = np.concatenate(rhs_ub)
    n_ineq = A_ub.shape[0]

    # Build PSD cone rows for [[P, X; X^T, Q]] upper-triangle vectorised
    sdp_size = nb + min_bpb
    sdp_vlen = sdp_size * (sdp_size + 1) // 2
    psd_r, psd_c, psd_v = [], [], []
    idx = 0
    for j in range(sdp_size):
        for i in range(j + 1):
            scale = 1.0 if i == j else np.sqrt(2)
            if i < nb and j < nb:
                psd_r.append(idx); psd_c.append(ix_P + j*nb + i); psd_v.append(-scale)
            elif i >= nb and j >= nb:
                psd_r.append(idx)
                psd_c.append(ix_Q + (j-nb)*min_bpb + (i-nb))
                psd_v.append(-scale)
            elif i < nb and j >= nb:
                xl = i * min_bpb + (j - nb)
                if xl < n_unif:
                    psd_r.append(idx); psd_c.append(xl); psd_v.append(-scale)
            idx += 1
    A_psd = sp.csc_matrix((psd_v, (psd_r, psd_c)), shape=(sdp_vlen, n_vars))

    A_full = sp.vstack([A_ub, A_psd], format='csc').astype(np.float64)
    b_full = np.concatenate([b_ub, np.zeros(sdp_vlen)]).astype(np.float64)
    print(f'  A_full: {A_full.shape}  nnz={A_full.nnz:,}  SDP size={sdp_size}')

    data   = {'c': c.astype(np.float64), 'A': A_full, 'b': b_full}
    cone   = {'l': n_ineq, 's': [sdp_size]}
    solver = scs.SCS(data, cone, eps=1e-4, max_iters=10000, verbose=False)
    sol    = solver.solve()

    elapsed = time.time() - t0
    xv = sol['x']
    x_opt = xv[:n] if xv is not None else None
    F1 = float(np.mean(np.maximum(D_PTV - A_ptv @ x_opt, 0))) if x_opt is not None else np.nan
    F2 = float(np.mean(np.maximum(A_esoph @ x_opt - D_ESOPH, 0))
             + np.mean(np.maximum(A_cord  @ x_opt - D_CORD,  0))) if x_opt is not None else np.nan
    t_nuc = float(xv[ix_tnuc]) if xv is not None else np.nan
    return {'x': x_opt, 'status': sol['info']['status'],
            'F1': F1, 'F2': F2, 'F_spectral': t_nuc,
            'solve_time': elapsed, 'label': f'Spectral lam={lam}'}

planner_bpb = [beamlets_per_beam[b] for b in beam_ids]
print(f'Beamlets per planner beam: {planner_bpb}')

print('Solving WITH spectral reg (lam=0.05)...')
r_spec_low = solve_with_spectral(
    A_ptv_sp, A_esoph_sp, A_cord_sp, A_lung_sp,
    n_beams=len(beam_ids), bpb=planner_bpb, lam=0.05)
print(f"  Status={r_spec_low['status']}  F1={r_spec_low['F1']:.4f}  "
      f"F2={r_spec_low['F2']:.4f}  Nuc={r_spec_low['F_spectral']:.4f}  "
      f"Time={r_spec_low['solve_time']:.1f}s")

print('Solving WITH spectral reg (lam=0.20)...')
r_spec_high = solve_with_spectral(
    A_ptv_sp, A_esoph_sp, A_cord_sp, A_lung_sp,
    n_beams=len(beam_ids), bpb=planner_bpb, lam=0.20)
print(f"  Status={r_spec_high['status']}  F1={r_spec_high['F1']:.4f}  "
      f"F2={r_spec_high['F2']:.4f}  Nuc={r_spec_high['F_spectral']:.4f}  "
      f"Time={r_spec_high['solve_time']:.1f}s")


## Clinical Metrics

In [ ]:
import pandas as pd
import os

def compute_metrics(x_opt, A_ptv, A_esoph, A_cord, A_lung):
    if x_opt is None:
        return {k: np.nan for k in ['d_ptv_mean','d_ptv_D95','underdose_frac',
                'd_esoph_max','esoph_V34','d_cord_max','cord_V45',
                'd_lung_mean','lung_V20','F1','F2']}
    d_ptv   = np.asarray(A_ptv   @ x_opt).flatten()
    d_esoph = np.asarray(A_esoph @ x_opt).flatten()
    d_cord  = np.asarray(A_cord  @ x_opt).flatten()
    d_lung  = np.asarray(A_lung  @ x_opt).flatten()
    return {
        'd_ptv_mean'     : float(np.mean(d_ptv)),
        'd_ptv_D95'      : float(np.percentile(d_ptv, 5)),
        'd_ptv_D99'      : float(np.percentile(d_ptv, 1)),
        'underdose_frac' : float(np.mean(d_ptv < D_PTV)),
        'd_esoph_max'    : float(np.max(d_esoph)),
        'esoph_V34'      : float(100 * np.mean(d_esoph > D_ESOPH)),
        'd_cord_max'     : float(np.max(d_cord)),
        'cord_V45'       : float(100 * np.mean(d_cord > D_CORD)),
        'd_lung_mean'    : float(np.mean(d_lung)),
        'lung_V20'       : float(100 * np.mean(d_lung > D_LUNG)),
        'F1'             : float(np.mean(np.maximum(D_PTV - d_ptv, 0))),
        'F2'             : float(np.mean(np.maximum(d_esoph - D_ESOPH, 0))
                               + np.mean(np.maximum(d_cord - D_CORD, 0))),
    }

all_results = {
    'No Spectral Reg'    : r_no_spec,
    'Spectral (lam=0.05)': r_spec_low,
    'Spectral (lam=0.20)': r_spec_high,
}

all_metrics = {}
for name, res in all_results.items():
    m = compute_metrics(res['x'], A_ptv_sp, A_esoph_sp, A_cord_sp, A_lung_sp)
    m['F_spectral'] = res.get('F_spectral', 0.0)
    m['solve_time']  = res['solve_time']
    all_metrics[name] = m

df = pd.DataFrame(all_metrics).T.round(3)
os.makedirs('results', exist_ok=True)
df.to_csv('results/metrics_table.csv')
display(df[['d_ptv_D95','underdose_frac','d_esoph_max','esoph_V34',
            'd_cord_max','d_lung_mean','F1','F2','solve_time']])


## DVH Comparison Plot

In [ ]:
import matplotlib.pyplot as plt

def dvh(dose_vec, n=200):
    d = np.linspace(0, dose_vec.max() * 1.05, n)
    return d, 100 * np.array([np.mean(dose_vec >= v) for v in d])

colours = ['#2196F3', '#FF5722', '#4CAF50']
structs = [('PTV', A_ptv_sp, D_PTV), ('Esophagus', A_esoph_sp, D_ESOPH),
           ('Cord', A_cord_sp, D_CORD), ('Lung', A_lung_sp, D_LUNG)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (sname, A_s, limit) in zip(axes.flat, structs):
    for (pname, res), c in zip(all_results.items(), colours):
        if res['x'] is not None:
            dv = np.asarray(A_s @ res['x']).flatten()
            d_ax, vf = dvh(dv)
            ax.plot(d_ax, vf, color=c, lw=2, label=pname)
    ax.axvline(limit, color='k', linestyle='--', lw=1.2, label=f'Limit {limit}Gy')
    ax.set_title(f'DVH — {sname}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Dose (Gy)'); ax.set_ylabel('Volume (%)')
    ax.legend(fontsize=8); ax.set_ylim(0, 105); ax.grid(True, alpha=0.3)

plt.suptitle('DVH: Convex Optimization With vs Without Spectral Regularization',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/dvh_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/dvh_comparison.png')


## Singular Value Spectra

Shows what spectral regularization actually does — it suppresses the higher singular values, forcing the fluence map matrix to be lower rank.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colours3 = ['#2196F3', '#FF5722', '#4CAF50']
for ax, (name, res), c in zip(axes, all_results.items(), colours3):
    xv = res['x']
    if xv is None:
        ax.set_title(f'{name} — FAILED'); continue
    min_bpb2 = min(planner_bpb); nb2 = len(beam_ids)
    X_mat = xv[:nb2 * min_bpb2].reshape(nb2, min_bpb2)
    svs   = np.linalg.svd(X_mat, compute_uv=False)
    ax.bar(range(1, len(svs) + 1), svs, color=c, alpha=0.85)
    ax.set_title(f'{name}\nNuclear Norm = {svs.sum():.2f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Singular value index')
    ax.set_ylabel('Magnitude')
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('SVD Spectra — Higher lambda Suppresses Large Singular Values (smoother plans)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/svd_spectra.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/svd_spectra.png')


## Lambda Sweep — Pareto Analysis

In [ ]:
lambdas = [0.0, 0.01, 0.05, 0.10, 0.20, 0.40]
sweep   = []
for lam in lambdas:
    print(f'  lam={lam:.2f}...', end=' ', flush=True)
    if lam == 0.0:
        res = solve_no_spectral(A_ptv_sp, A_esoph_sp, A_cord_sp, A_lung_sp)
    else:
        res = solve_with_spectral(A_ptv_sp, A_esoph_sp, A_cord_sp, A_lung_sp,
                                  n_beams=len(beam_ids), bpb=planner_bpb, lam=lam)
    m = compute_metrics(res['x'], A_ptv_sp, A_esoph_sp, A_cord_sp, A_lung_sp)
    m['lambda'] = lam; m['F_spectral'] = res.get('F_spectral', 0.0)
    sweep.append(m)
    print(f"D95={m['d_ptv_D95']:.1f}Gy  EsophMax={m['d_esoph_max']:.1f}Gy")

df_sweep = pd.DataFrame(sweep)
df_sweep.to_csv('results/lambda_sweep.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(df_sweep['lambda'], df_sweep['d_ptv_D95'],  'b-o', lw=2, label='D95 (PTV)')
axes[0].plot(df_sweep['lambda'], df_sweep['d_ptv_mean'], 'b--s', lw=2, label='D_mean (PTV)')
axes[0].axhline(D_PTV, color='gray', linestyle=':', label='60 Gy target')
axes[0].legend(); axes[0].set_xlabel('lambda')
axes[0].set_title('Tumor Coverage vs Lambda'); axes[0].grid(alpha=0.3)

axes[1].plot(df_sweep['lambda'], df_sweep['d_esoph_max'],  'r-o', lw=2, label='Esoph Max')
axes[1].plot(df_sweep['lambda'], df_sweep['d_cord_max'],   'r--s', lw=2, label='Cord Max')
axes[1].plot(df_sweep['lambda'], df_sweep['d_lung_mean'],  'm:^', lw=2, label='Lung Mean')
axes[1].axhline(D_ESOPH, color='r',  linestyle=':', alpha=0.4)
axes[1].axhline(D_CORD,  color='r',  linestyle=':', alpha=0.4)
axes[1].axhline(D_LUNG,  color='m',  linestyle=':', alpha=0.4)
axes[1].legend(); axes[1].set_xlabel('lambda')
axes[1].set_title('OAR Doses vs Lambda'); axes[1].grid(alpha=0.3)

axes[2].plot(df_sweep['F1'], df_sweep['F2'], 'ko-', lw=2, markersize=9)
for _, row in df_sweep.iterrows():
    axes[2].annotate(f"l={row['lambda']:.2f}", (row['F1'], row['F2']),
                     textcoords='offset points', xytext=(5, 5), fontsize=8)
axes[2].set_xlabel('F1: PTV Underdose')
axes[2].set_ylabel('F2: OAR Overdose')
axes[2].set_title('Pareto Frontier (F1 vs F2)'); axes[2].grid(alpha=0.3)

plt.suptitle('Effect of Spectral Regularization Strength (Lambda Sweep)', fontsize=13)
plt.tight_layout()
plt.savefig('results/lambda_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/lambda_sweep.png and results/lambda_sweep.csv')
